In [1]:
import jax.numpy as jnp
from loguru import logger

from thrml import SpinNode, Block # Block -> grouping of nodes whose state we can supply/access together.
from thrml.models import IsingEBM

In [2]:
s1 = SpinNode() # SpinNode = one binary random variable.
s2 = SpinNode()

nodes = [s1, s2] 
edges = [(s1, s2)]

biases = jnp.array([0.0, 0.0])
#weights = jnp.array([0.8])
weights = jnp.array([-0.8])
beta = jnp.array(1.0)

model = IsingEBM( # The actual energy-based model.
    nodes=nodes,
    edges=edges,
    biases=biases,
    weights=weights,
    beta=beta,
)

logger.info("Created a 2-spin Ising model")
logger.info(f"Nodes: {len(model.nodes)}")
logger.info(f"Edges: {len(model.edges)}")
logger.info(f"Weights: {model.weights}")

2026-10-27 17:01:12.921 | INFO     | __main__:<module>:20 - Created a 2-spin Ising model
2026-10-27 17:01:12.922 | INFO     | __main__:<module>:21 - Nodes: 2
2026-10-27 17:01:12.924 | INFO     | __main__:<module>:22 - Edges: 1
2026-10-27 17:01:12.925 | INFO     | __main__:<module>:23 - Weights: [-0.8]


In [3]:
block = Block(nodes)

states = [
    jnp.array([False, False]),
    jnp.array([False, True]),
    jnp.array([True, False]),
    jnp.array([True, True]),
]

for state in states:
    energy = model.energy( # evaluate E(s) at each state
        state=[state],
        blocks=[block],
    )

    logger.info(f"state={state}  energy={energy}")

2026-10-27 17:01:13.786 | INFO     | __main__:<module>:16 - state=[False False]  energy=0.800000011920929
2026-10-27 17:01:13.792 | INFO     | __main__:<module>:16 - state=[False  True]  energy=-0.800000011920929
2026-10-27 17:01:13.797 | INFO     | __main__:<module>:16 - state=[ True False]  energy=-0.800000011920929
2026-10-27 17:01:13.803 | INFO     | __main__:<module>:16 - state=[ True  True]  energy=0.800000011920929


In [4]:
model = IsingEBM(
    nodes=nodes,
    edges=edges,
    biases=jnp.array([0.0, 0.0]),
    weights=jnp.array([0.8]),
    beta=jnp.array(1.0),
)

states = jnp.array([
    [False, False],
    [False, True],
    [True, False],
    [True, True],
])

block = Block(nodes)

energies = jnp.array([
    model.energy([state], [block])
    for state in states
])

scores = jnp.exp(-energies)
Z = jnp.sum(scores)
probabilities = scores / Z

logger.info(f"Partition function Z = {Z:.4f}")

for state, energy, prob in zip(states, energies, probabilities):
    logger.info(f"{state} | Energy: {energy:.2f} | Probability: {prob:.4f}")


2026-10-27 17:01:13.990 | INFO     | __main__:<module>:27 - Partition function Z = 5.3497
2026-10-27 17:01:14.047 | INFO     | __main__:<module>:30 - [False False] | Energy: -0.80 | Probability: 0.4160
2026-10-27 17:01:14.049 | INFO     | __main__:<module>:30 - [False  True] | Energy: 0.80 | Probability: 0.0840
2026-10-27 17:01:14.050 | INFO     | __main__:<module>:30 - [ True False] | Energy: 0.80 | Probability: 0.0840
2026-10-27 17:01:14.052 | INFO     | __main__:<module>:30 - [ True  True] | Energy: -0.80 | Probability: 0.4160


In [5]:
# compute the expectation
spins = 2 * states.astype(jnp.int32) - 1 # converts T & F into 1 & -1. maps to 1 & 0 then uses 2x-1 to 1 & -1

products = spins[:, 0] * spins[:, 1] # multiplies two constraint functions together or s_1*s_2

joint_expectation = jnp.sum(probabilities * products)

logger.info(f"E[s1*s2] = {joint_expectation:.6f}")


2026-10-27 17:01:14.349 | INFO     | __main__:<module>:8 - E[s1*s2] = 0.664037


In [6]:

for J in [-0.8, 0.0, 0.8]:

    model = IsingEBM(
        nodes=nodes,
        edges=edges,
        biases=jnp.array([0.0, 0.0]),
        weights=jnp.array([J]),
        beta=jnp.array(1.0),
    )

    energies = jnp.array([
        model.energy([state], [block])
        for state in states
    ])

    scores = jnp.exp(-energies)
    probabilities = scores / jnp.sum(scores)

    spins = 2 * states.astype(jnp.int32) - 1
    products = spins[:, 0] * spins[:, 1]

    eta = jnp.sum(probabilities * products)

    logger.info(f"J = {J:+.1f} | E[S1*S2] = {eta:.4f}")


2026-10-27 17:01:14.836 | INFO     | __main__:<module>:24 - J = -0.8 | E[S1*S2] = -0.6640
2026-10-27 17:01:14.852 | INFO     | __main__:<module>:24 - J = +0.0 | E[S1*S2] = 0.0000
2026-10-27 17:01:14.868 | INFO     | __main__:<module>:24 - J = +0.8 | E[S1*S2] = 0.6640
